For each NACE Class get the 100 chunks that scored highest across all the reports 

In [35]:
import pandas as pd
import glob
import os
import tqdm
import numpy as np
import matplotlib.pyplot as plt
os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
from test_base import *

In [36]:
os.chdir('/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis')

In [37]:
overview_path = "data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = "data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_overview.csv"

In [38]:
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"
raw_data_path = "results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"
raw_data_path = "results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"

In [79]:
recordings = pd.read_csv(raw_data_path + "recordings.csv")

In [80]:
recordings

,Unnamed: 0,name,NACE,label_description,nace_lvl_1,position_lvl_1,classes_lvl_1,classification_lvl_1,first_class_1,position_lvl_2,classes_lvl_2,classification_lvl_2,first_class_2,position_lvl_3,classes_lvl_3,classification_lvl_3,first_class_3
0,0,Brimstone Investment Corporation Limited1.txt,3.11,Marine fishing,A,5,21,{'K_FINANCIAL AND INSURANCE ACTIVITIES': 0.033...,K,4,88,"{'64_Financial service activities, except insu...",6,25,272,"{'64.3_Trusts, funds and similar financial ent...",6
1,1,Gigante Salmon AS1.txt,3.11,Marine fishing,A,1,21,{'K_FINANCIAL AND INSURANCE ACTIVITIES': 0.012...,K,0,88,"{'3_Fishing and aquaculture': 0.0455, '70_Acti...",3,4,272,"{'10.2_Processing and preserving of fish, crus...",1
2,2,Sapmer SA2.txt,3.11,Marine fishing,A,3,21,{'U_ACTIVITIES OF EXTRATERRITORIAL ORGANISATIO...,U,11,88,{'97_Activities of households as employers of ...,9,47,272,"{'01.5_Mixed farming': 0.017, '64.1_Monetary i...",0
3,3,Sea Harvest Group Ltd.1.txt,3.11,Marine fishing,A,0,21,"{'A_AGRICULTURE, FORESTRY AND FISHING': 0.0171...",A,0,88,{'3_Fishing and aquaculture': 0.08299999999999...,3,1,272,"{'03.2_Aquaculture': 0.088, '03.1_Fishing': 0....",0


In [39]:
reports = glob.glob(raw_data_path + "*/*_short.csv")
reports = glob.glob(raw_data_path + "*/*_long.csv")
len(reports)

3217

In [40]:
#sample_ratio = 1

In [41]:
#max_elements_per_class = 1000000
#top_k_sentences = 200000

In [42]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
#filter_only_right_chunks = True

In [43]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
#with_null_classifiers = True

In [44]:
new_threshold_cos_sin = 0.35

In [45]:
nace_level_descriptions = 3
nace_level = 3
assert nace_level_descriptions >= nace_level

In [46]:
training_data_path = "data/training_data/approach_2"

In [47]:
#suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}"
suffix = f"2nd_approach" + f"__nace_level_{nace_level}__cos_thres_{new_threshold_cos_sin}"

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path
os.makedirs(end_path, exist_ok=True)

In [48]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path, index_col=0)
df_overview.head()

,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,NACE,...,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report,NACE_lvl_3
0,CA05335P1099,Auxly Cannabis Group Inc.,1,1987.0,CAN,05335P109,19960301.0,CAN,003V9K-E,1.30,...,BDGMQB,Auxly Cannabis Group Inc.,1.0,XLY-CA,1,SHARE,BDGMQB3,A,Auxly Cannabis Group Inc.2.pdf,NaN
1,JP3947800003,"MEGMILK SNOW BRAND Co., Ltd.",1,2009.0,JPN,J41966102,20220727.0,JPN,0833Y1-E,1.41,...,B3ZC07,"MEGMILK SNOW BRAND Co., Ltd.",1.0,MMSBF-US,0,SHARE,BKQN701,A,"MEGMILK SNOW BRAND Co., Ltd.2.pdf",NaN
2,ID1000167901,PT Cilacap Samudera Fishing Industry Tbk,0,1999.0,IDN,Y129H6108,20220527.0,IDN,@NA,3.12,...,BMBMZG,PT Cilacap Samudera Fishing Industry Tbk,1.0,ASHA-ID,0,SHARE,BMBMZG9,A,PT Cilacap Samudera Fishing Industry Tbk3.pdf,NaN
3,JP3843250006,Hokuto Corporation,1,1964.0,JPN,J2224T102,19941202.0,JPN,05HY7N-E,1.30,...,643271,Hokuto Corporation,1.0,1379-JP,1,SHARE,6432715,A,Hokuto Corporation1.pdf,NaN
4,VN000000VTQ6,Viet Trung Quang Binh Joint Stock Co,0,1961.0,VNM,Y937XS108,NaN,VNM,@NA,2.30,...,BMCR2W,Viet Trung Quang Binh Joint Stock Co,1.0,VTQ-VN,0,SHARE,BMCR2W8,A,Viet Trung Quang Binh Joint Stock Co3.pdf,NaN


In [49]:
df_nace_codes_descriptions = pd.read_csv("data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

### For each class c: those paragraphs p of reports in class c with cos-sim(p, c) > 0.5

In [50]:
# get the NACE code from a column name like "Scores_1_Crop and animal..." or "Scores_A_Crop and animal..."
get_nace_code_from_column_name = lambda col_name : col_name.split("_")[1]

In [96]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    
    report_name = os.path.basename(report).replace(".txt_long.csv", "") + ".pdf"
    if not len(df_overview[df_overview["Report"]==report_name]): 
        continue
    report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
    report_code = get_all_level(report_code)[nace_level]

    if report_code == "21.1":
        print(report_code)
    else:
        continue
    df = pd.read_csv(report)

    scores = [c for c in df.columns if "Scores" in c]
    
    df["max_class_sim"] = [get_nace_code_from_column_name(scores[i]) for i in np.argmax(df[scores], 1)]
    df["Score"] = df[scores].max(1)
    
    df.loc[(df["max_class_sim"] == report_code) & (df["Score"] > new_threshold_cos_sin), "NACE_Code"] = report_code
    df["NACE_Code"] = df["NACE_Code"].fillna("NO_CLASS")

    # print()
    # print(df["max_class_sim"].value_counts())
    # print(report_code)
    # print(len(df[(df["max_class_sim"] == report_code) & (df["Score"] > new_threshold_cos_sin)]))

    result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])

  0%|                                                                                                                                                                                        | 0/3217 [00:00<?, ?it/s]/tmp/ipykernel_689108/2385339020.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])
  4%|██████▊                                                                                                                                                                      | 126/3217 [00:00<00:04, 653.64it/s]

21.1
21.1
21.1


 25%|██████████████████████████████████████████▍                                                                                                                                 | 794/3217 [00:00<00:02, 1108.40it/s]

21.1
21.1


 33%|█████████████████████████████████████████████████████████▏                                                                                                                 | 1076/3217 [00:01<00:02, 1047.46it/s]

21.1
21.1
21.1


 55%|██████████████████████████████████████████████████████████████████████████████████████████████▋                                                                            | 1782/3217 [00:01<00:01, 1153.79it/s]

21.1
21.1
21.1
21.1
21.1


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3217/3217 [00:02<00:00, 1252.93it/s]


In [87]:
stats = result.groupby("NACE_Code").count()["Sentences"]
stats

NACE_Code
21.1           14
NO_CLASS    10626
Name: Sentences, dtype: int64

In [67]:
amount_no_class = stats[stats.index != "NO_CLASS"].max().item()
amount_no_class

10919

In [68]:
result_right = result[result["NACE_Code"] != "NO_CLASS"]
result_NO_CLASS = result.loc[result["NACE_Code"] == "NO_CLASS"].sample(n=amount_no_class)
result_final = pd.concat([result_right, result_NO_CLASS], axis=0)

In [69]:
stats = result_final.groupby("NACE_Code").agg({
    "Sentences": "count", 
    "Score": "mean"
})
stats

,Sentences,Score
NACE_Code,,
01.1,43,0.418227
01.2,1,0.436468
01.3,23,0.420140
01.4,9,0.422950
01.5,191,0.476353
...,...,...
93.1,508,0.441864
93.2,402,0.439247
95.1,21,0.501802


In [70]:
stats.to_csv(end_path + "/statistics.csv")

In [71]:
# recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

# df_recordings = pd.DataFrame(recordings)
# df_recordings = df_recordings.sort_values(by="Code")
# df_recordings.head()

#df_recordings.to_csv(end_path + "/statistics.csv")

In [72]:
full_df= result_final.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df["Evaluation"] = None
full_df["Notes"] = None

In [73]:
# Store each class for reading
n = 40
for nace_class in stats.index: 
    # print(nace_class)
    temp = full_df[full_df["NACE_Code"] == nace_class].copy()
    temp["Evaluation"] = None
    temp["Notes"] = None
    if len(temp) >= n:
        temp = temp.sample(n=40)
    temp.to_csv(os.path.join(end_path, nace_class + ".csv"))

In [74]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 49205, Test size: 16402, Validation size: 16402


In [75]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [76]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [77]:
end_path

'data/training_data/approach_2/dataset__reports_subset_from_full_data_2_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3__2nd_approach__nace_level_3__cos_thres_0.35'

In [78]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'